# PCB Orientation Detection - CNN Training
## Clean Training with Best Hyperparameters and Fixed Label Assignment

**Objective**: Train a CNN model for PCB Pass/Fail classification using best hyperparameters, fix label inversion issues, and export for live classification.

In [1]:
# Section 1: Import Required Libraries
import PIL
import PIL.Image
import tensorflow as tf
import pathlib
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

print("✓ All libraries imported successfully")

# Setup data directory and parameters
data_dir = pathlib.Path('Data/Processed_data')
img_height = 244
img_width = 244
batch_size = 6

print(f"\n✓ Data directory: {data_dir}")
print(f"✓ Image dimensions: {img_height}x{img_width}")
print(f"✓ Batch size: {batch_size}")

✓ All libraries imported successfully

✓ Data directory: Data\Processed_data
✓ Image dimensions: 244x244
✓ Batch size: 6


In [2]:
# Section 2: Load and Prepare Data with EXPLICIT Label Mapping
print("\n" + "="*80)
print("LOADING DATA WITH EXPLICIT LABEL ASSIGNMENT")
print("="*80)

# Check available data
image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"Total images found: {image_count}")

# Load using tf.keras API - respects alphabetical directory order
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size)

print(f"\n✓ Training dataset created")
print(f"✓ Validation dataset created")
print(f"✓ Class names from directory: {train_ds.class_names}")
print("\nIMPORTANT: Class names are ordered ALPHABETICALLY")
print(f"  Index 0 → {train_ds.class_names[0]}")
print(f"  Index 1 → {train_ds.class_names[1]}")


LOADING DATA WITH EXPLICIT LABEL ASSIGNMENT
Total images found: 7515
Found 7515 files belonging to 2 classes.
Using 5261 files for training.
Found 7515 files belonging to 2 classes.
Using 2254 files for validation.

✓ Training dataset created
✓ Validation dataset created
✓ Class names from directory: ['Fail_data', 'Pass_data']

IMPORTANT: Class names are ordered ALPHABETICALLY
  Index 0 → Fail_data
  Index 1 → Pass_data


In [3]:
# Section 3: Build CNN Model with Best Hyperparameters
print("\n" + "="*80)
print("BUILDING CNN MODEL WITH BEST HYPERPARAMETERS")
print("="*80)

# Best hyperparameters from tuning (Config 5 - 93.33% accuracy)
filters_base = 64
dropout_rate = 0.25
learning_rate = 0.0005
num_classes = 2

print(f"\n✓ BEST HYPERPARAMETERS (from tuning):")
print(f"  - Filters Base: {filters_base}")
print(f"  - Dropout Rate: {dropout_rate}")
print(f"  - Learning Rate: {learning_rate}")
print(f"  - Expected Fold Accuracy: 93.33%")

# Create export directory
os.makedirs('Export', exist_ok=True)

# Build model with best hyperparameters
print(f"\nBuilding CNN architecture...")
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),
    
    # Block 1
    tf.keras.layers.Conv2D(filters_base, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(dropout_rate),
    
    # Block 2
    tf.keras.layers.Conv2D(filters_base * 2, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(dropout_rate),
    
    # Block 3
    tf.keras.layers.Conv2D(filters_base * 4, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(dropout_rate),
    
    # Dense layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(dropout_rate),
    tf.keras.layers.Dense(num_classes)  # NO ACTIVATION - returns raw logits
])

print("✓ Model architecture created")
print("✓ CRITICAL: Output layer has NO activation (returns raw logits)")


BUILDING CNN MODEL WITH BEST HYPERPARAMETERS

✓ BEST HYPERPARAMETERS (from tuning):
  - Filters Base: 64
  - Dropout Rate: 0.25
  - Learning Rate: 0.0005
  - Expected Fold Accuracy: 93.33%

Building CNN architecture...
✓ Model architecture created
✓ CRITICAL: Output layer has NO activation (returns raw logits)


In [4]:
# Section 4: Compile Model
print("\n" + "="*80)
print("COMPILING MODEL")
print("="*80)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("✓ Model compiled successfully")
print(f"  - Optimizer: Adam (lr={learning_rate})")
print(f"  - Loss: SparseCategoricalCrossentropy(from_logits=True)")
print(f"  - Metrics: accuracy")
print(f"\n✓ Configuration is CORRECT:")
print(f"  - Output layer: NO activation (returns raw logits)")
print(f"  - Loss expects: from_logits=True")
print(f"  - This ensures proper alignment between loss and network output")


COMPILING MODEL
✓ Model compiled successfully
  - Optimizer: Adam (lr=0.0005)
  - Loss: SparseCategoricalCrossentropy(from_logits=True)
  - Metrics: accuracy

✓ Configuration is CORRECT:
  - Output layer: NO activation (returns raw logits)
  - Loss expects: from_logits=True
  - This ensures proper alignment between loss and network output


In [5]:
# Section 5: Verify Class Mapping
print("\n" + "="*80)
print("CLASS MAPPING VERIFICATION")
print("="*80)

print(f"\nClass names from train_ds: {train_ds.class_names}")
print(f"Number of classes: {len(train_ds.class_names)}")

# Create mapping dictionary
class_mapping = {i: name for i, name in enumerate(train_ds.class_names)}
print(f"\nClass mapping in model:")
for idx, class_name in class_mapping.items():
    print(f"  Index {idx} → {class_name}")

# Verify class distribution in training data
print(f"\nAnalyzing class distribution in training data...")
train_labels = []
for images, labels in train_ds:
    train_labels.extend(labels.numpy())
train_labels = np.array(train_labels)

unique, counts = np.unique(train_labels, return_counts=True)
print(f"\nClass distribution in training data:")
for class_idx in range(len(train_ds.class_names)):
    count = counts[class_idx] if class_idx in unique else 0
    print(f"  Index {class_idx} ({train_ds.class_names[class_idx]}): {count} samples")

print(f"\n✓ CLASS MAPPING CONFIRMED:")
print(f"  Expected: Index 0 = Fail_data, Index 1 = Pass_data")
if train_ds.class_names[0] == 'Fail_data' and train_ds.class_names[1] == 'Pass_data':
    print(f"  ✓ CORRECT - Using alphabetical order from directories")
else:
    print(f"  ⚠ WARNING - Order may be different, verify with predictions")


CLASS MAPPING VERIFICATION

Class names from train_ds: ['Fail_data', 'Pass_data']
Number of classes: 2

Class mapping in model:
  Index 0 → Fail_data
  Index 1 → Pass_data

Analyzing class distribution in training data...

Class distribution in training data:
  Index 0 (Fail_data): 4830 samples
  Index 1 (Pass_data): 431 samples

✓ CLASS MAPPING CONFIRMED:
  Expected: Index 0 = Fail_data, Index 1 = Pass_data
  ✓ CORRECT - Using alphabetical order from directories


In [6]:
# Section 6: Train Model
print("\n" + "="*80)
print("TRAINING MODEL")
print("="*80)

# Define callback to stop at 90% accuracy
class AccuracyThresholdCallback(tf.keras.callbacks.Callback):
    def __init__(self, threshold=0.90):
        super().__init__()
        self.threshold = threshold
    
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_accuracy = logs.get('val_accuracy')
        if val_accuracy is not None and val_accuracy >= self.threshold:
            print(f"\n✓ Validation accuracy {val_accuracy:.4f} >= {self.threshold:.4f}")
            print("✓ Stopping training early!")
            self.model.stop_training = True

# Train model with callbacks
print("\nStarting training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        AccuracyThresholdCallback(threshold=0.90)
    ]
)

print("\n✓ Training complete!")
print(f"\nTraining history:")
print(f"  Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"  Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"  Final training loss: {history.history['loss'][-1]:.4f}")
print(f"  Final validation loss: {history.history['val_loss'][-1]:.4f}")


TRAINING MODEL

Starting training...
Epoch 1/10
877/877 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7630 - loss: 0.5371
✓ Validation accuracy 0.9476 >= 0.9000
✓ Stopping training early!
877/877 ━━━━━━━━━━━━━━━━━━━━ 1355s 2s/step - accuracy: 0.8569 - loss: 0.3558 - val_accuracy: 0.9476 - val_loss: 0.1811
Restoring model weights from the end of the best epoch: 1.

✓ Training complete!

Training history:
  Final training accuracy: 0.8569
  Final validation accuracy: 0.9476
  Final training loss: 0.3558
  Final validation loss: 0.1811


In [7]:
# Print model summary
print("\nModel Summary:")
model.summary()

# Evaluate on validation set
print("\n" + "="*80)
print("FINAL EVALUATION ON VALIDATION SET")
print("="*80)
val_loss, val_accuracy = model.evaluate(val_ds)
print(f"\nValidation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")


Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 244, 244, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 244, 244, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 244, 244, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 122, 122, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 122, 122, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 122, 122, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 122, 122, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 61, 61, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 61, 61, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 61, 61, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 61, 61, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 30, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 230400)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    29,491,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,591,816 (341.77 MB)

 Trainable params: 29,863,554 (113.92 MB)

 Non-trainable params: 1,152 (4.50 KB)

 Optimizer params: 59,727,110 (227.84 MB)


FINAL EVALUATION ON VALIDATION SET
376/376 ━━━━━━━━━━━━━━━━━━━━ 77s 204ms/step - accuracy: 0.9476 - loss: 0.1811

Validation Loss: 0.1811
Validation Accuracy: 0.9476


In [8]:
# Section 7: Test Predictions on Known Sample Images
print("\n" + "="*80)
print("TESTING PREDICTIONS ON KNOWN SAMPLE IMAGES")
print("="*80)

# Test on a PASS image
pass_img_path = list(data_dir.glob('Pass_data/*.jpg'))[0]
print(f"\n--- Testing on PASS image: {pass_img_path.name} ---")

pass_img = tf.keras.utils.load_img(pass_img_path, target_size=(img_height, img_width))
pass_array = tf.keras.utils.img_to_array(pass_img)
pass_array = np.expand_dims(pass_array, axis=0)

# Get raw logits from model
pass_logits = model.predict(pass_array, verbose=0)
print(f"Raw logits: {pass_logits[0]}")

# Convert logits to probabilities using softmax
pass_probs = tf.nn.softmax(pass_logits[0]).numpy()
print(f"Softmax probabilities: {pass_probs}")

pass_pred_idx = np.argmax(pass_probs)
pass_pred_label = train_ds.class_names[pass_pred_idx]
pass_confidence = pass_probs[pass_pred_idx]

print(f"Predicted index: {pass_pred_idx}")
print(f"Predicted class: {pass_pred_label}")
print(f"Confidence: {pass_confidence:.4f} ({pass_confidence*100:.2f}%)")

# Test on a FAIL image
fail_img_path = list(data_dir.glob('Fail_data/*.jpg'))[0]
print(f"\n--- Testing on FAIL image: {fail_img_path.name} ---")

fail_img = tf.keras.utils.load_img(fail_img_path, target_size=(img_height, img_width))
fail_array = tf.keras.utils.img_to_array(fail_img)
fail_array = np.expand_dims(fail_array, axis=0)

# Get raw logits from model
fail_logits = model.predict(fail_array, verbose=0)
print(f"Raw logits: {fail_logits[0]}")

# Convert logits to probabilities using softmax
fail_probs = tf.nn.softmax(fail_logits[0]).numpy()
print(f"Softmax probabilities: {fail_probs}")

fail_pred_idx = np.argmax(fail_probs)
fail_pred_label = train_ds.class_names[fail_pred_idx]
fail_confidence = fail_probs[fail_pred_idx]

print(f"Predicted index: {fail_pred_idx}")
print(f"Predicted class: {fail_pred_label}")
print(f"Confidence: {fail_confidence:.4f} ({fail_confidence*100:.2f}%)")

# Verify predictions are CORRECT
print("\n" + "="*80)
print("PREDICTION VERIFICATION")
print("="*80)

pass_correct = pass_pred_idx == 1 and train_ds.class_names[1] == 'Pass_data'
fail_correct = fail_pred_idx == 0 and train_ds.class_names[0] == 'Fail_data'

print(f"\nPass image prediction correct: {pass_correct}")
print(f"  Expected: Index 1 (Pass_data)")
print(f"  Actual: Index {pass_pred_idx} ({pass_pred_label})")
print(f"  Confidence: {pass_confidence*100:.2f}%")

print(f"\nFail image prediction correct: {fail_correct}")
print(f"  Expected: Index 0 (Fail_data)")
print(f"  Actual: Index {fail_pred_idx} ({fail_pred_label})")
print(f"  Confidence: {fail_confidence*100:.2f}%")

if pass_correct and fail_correct:
    print("\n✓✓✓ EXCELLENT! Model predictions are CORRECT! ✓✓✓")
else:
    print("\n⚠ WARNING: Predictions may be incorrect or inverted!")
    print("  Review the class mapping and label assignment")


TESTING PREDICTIONS ON KNOWN SAMPLE IMAGES

--- Testing on PASS image: Lobbypass_00001.jpg ---
Raw logits: [-0.1848936  0.057833 ]
Softmax probabilities: [0.43961456 0.56038547]
Predicted index: 1
Predicted class: Pass_data
Confidence: 0.5604 (56.04%)

--- Testing on FAIL image: Lobbyfail10_00001.jpg ---
Raw logits: [ 0.830123 -0.81443 ]
Softmax probabilities: [0.8381535  0.16184649]
Predicted index: 0
Predicted class: Fail_data
Confidence: 0.8382 (83.82%)

PREDICTION VERIFICATION

Pass image prediction correct: True
  Expected: Index 1 (Pass_data)
  Actual: Index 1 (Pass_data)
  Confidence: 56.04%

Fail image prediction correct: True
  Expected: Index 0 (Fail_data)
  Actual: Index 0 (Fail_data)
  Confidence: 83.82%

✓✓✓ EXCELLENT! Model predictions are CORRECT! ✓✓✓


In [9]:
# Section 8: Export Model and Class Names
print("\n" + "="*80)
print("EXPORTING MODEL AND METADATA")
print("="*80)

# Create export directory
os.makedirs('Export', exist_ok=True)

# Save trained model
model_path = 'Export/ot_model.keras'
print(f"\nSaving model to {model_path}...")
model.save(model_path)
print(f"✓ Model saved successfully")

# Verify model loads correctly
print(f"\nVerifying saved model...")
loaded_model = tf.keras.models.load_model(model_path)
print(f"✓ Model loads successfully")
print(f"✓ Output shape: {loaded_model.output_shape}")

# Save class names and mapping for live_classification.py
class_names_info = {
    "class_names": train_ds.class_names,
    "class_mapping": {i: name for i, name in enumerate(train_ds.class_names)},
    "note": "Use these exact class names in live_classification.py"
}

class_names_file = "Export/class_names.json"
with open(class_names_file, 'w') as f:
    json.dump(class_names_info, f, indent=2)

print(f"\n✓ Class names saved to {class_names_file}")
print(f"Class mapping:")
for idx, name in enumerate(train_ds.class_names):
    print(f"  Index {idx} → {name}")

# Display export summary
print(f"\n" + "="*80)
print("EXPORT SUMMARY")
print("="*80)
print(f"\n✓ Model exported to: Export/ot_model.keras")
print(f"✓ Class names exported to: Export/class_names.json")
print(f"✓ Ready for live classification!")


EXPORTING MODEL AND METADATA

Saving model to Export/ot_model.keras...
✓ Model saved successfully

Verifying saved model...
✓ Model loads successfully
✓ Output shape: (None, 2)

✓ Class names saved to Export/class_names.json
Class mapping:
  Index 0 → Fail_data
  Index 1 → Pass_data

EXPORT SUMMARY

✓ Model exported to: Export/ot_model.keras
✓ Class names exported to: Export/class_names.json
✓ Ready for live classification!


In [10]:
# Section 9: Verify Exported Model Works Correctly
print("\n" + "="*80)
print("VERIFYING EXPORTED MODEL")
print("="*80)

# Load exported model
print("\nLoading exported model from Export/ot_model.keras...")
exported_model = tf.keras.models.load_model('Export/ot_model.keras')
print("✓ Model loaded successfully")

# Test on Pass image
print(f"\n--- Testing exported model on PASS image ---")
pass_img_path = list(data_dir.glob('Pass_data/*.jpg'))[1]  # Use different image
print(f"Image: {pass_img_path.name}")

pass_img = tf.keras.utils.load_img(pass_img_path, target_size=(img_height, img_width))
pass_array = tf.keras.utils.img_to_array(pass_img)
pass_array = np.expand_dims(pass_array, axis=0)

pass_logits = exported_model.predict(pass_array, verbose=0)
pass_probs = tf.nn.softmax(pass_logits[0]).numpy()
pass_pred_idx = np.argmax(pass_probs)

print(f"Predicted: {train_ds.class_names[pass_pred_idx]} (index {pass_pred_idx}, confidence {pass_probs[pass_pred_idx]*100:.2f}%)")

# Test on Fail image
print(f"\n--- Testing exported model on FAIL image ---")
fail_img_path = list(data_dir.glob('Fail_data/*.jpg'))[1]  # Use different image
print(f"Image: {fail_img_path.name}")

fail_img = tf.keras.utils.load_img(fail_img_path, target_size=(img_height, img_width))
fail_array = tf.keras.utils.img_to_array(fail_img)
fail_array = np.expand_dims(fail_array, axis=0)

fail_logits = exported_model.predict(fail_array, verbose=0)
fail_probs = tf.nn.softmax(fail_logits[0]).numpy()
fail_pred_idx = np.argmax(fail_probs)

print(f"Predicted: {train_ds.class_names[fail_pred_idx]} (index {fail_pred_idx}, confidence {fail_probs[fail_pred_idx]*100:.2f}%)")

# Final summary
print("\n" + "="*80)
print("TRAINING AND EXPORT COMPLETE!")
print("="*80)
print(f"\n✓ Model trained with best hyperparameters:")
print(f"  - Filters: {filters_base}, Dropout: {dropout_rate}, LR: {learning_rate}")
print(f"\n✓ Label mapping fixed and verified:")
print(f"  - Index 0 → {train_ds.class_names[0]}")
print(f"  - Index 1 → {train_ds.class_names[1]}")
print(f"\n✓ Model exported to:")
print(f"  - Export/ot_model.keras")
print(f"  - Export/class_names.json")
print(f"\n✓ Next step: Run live_classification.py to verify predictions")
print("="*80)


VERIFYING EXPORTED MODEL

Loading exported model from Export/ot_model.keras...
✓ Model loaded successfully

--- Testing exported model on PASS image ---
Image: Lobbypass_00002.jpg
Predicted: Pass_data (index 1, confidence 59.32%)

--- Testing exported model on FAIL image ---
Image: Lobbyfail10_00002.jpg
Predicted: Fail_data (index 0, confidence 76.48%)

TRAINING AND EXPORT COMPLETE!

✓ Model trained with best hyperparameters:
  - Filters: 64, Dropout: 0.25, LR: 0.0005

✓ Label mapping fixed and verified:
  - Index 0 → Fail_data
  - Index 1 → Pass_data

✓ Model exported to:
  - Export/ot_model.keras
  - Export/class_names.json

✓ Next step: Run live_classification.py to verify predictions
